# 🎬 Sentiment Analysis of IMDB Movie Reviews using an RNN (PyTorch)

**Goal:** Build a Recurrent Neural Network (RNN) that reads a movie review and predicts whether the review is **positive** or **negative**.

**What this notebook does, step by step**

1. Load the IMDB dataset (50,000 movie reviews)
2. Check and clean the data
3. Pre-process the text (lowercase, remove URLs / punctuation / HTML / stopwords, stemming)
4. Convert text into numbers (label encoding + TF-IDF)
5. Split into train and test sets and create PyTorch data loaders
6. Build, train and evaluate the RNN

**Tech stack:** Python · Pandas · NLTK · Scikit-learn · PyTorch

---
### 📑 Table of Contents
1. Import Libraries
2. Load the Dataset
3. Data Checks and Cleaning
4. Text Pre-processing
5. Train/Test Split and Data Loaders
6. Build the RNN Model
7. Train the Model
8. Evaluate the Model
9. Conclusion

## 1. Import Libraries

All the libraries used in this project are imported once here, so the rest of the notebook stays clean:

- **pandas** – load and handle the dataset
- **re** – regular expressions for text cleaning
- **nltk** – tokenization, stopwords and stemming
- **scikit-learn** – label encoding, TF-IDF and train/test split
- **torch** – build and train the RNN
- **matplotlib** – plot a chart of the data

In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## 2. Load the Dataset

The IMDB dataset has two columns:

| Column | Meaning |
|---|---|
| `review` | The text of the movie review |
| `sentiment` | The label: `positive` or `negative` |

In [33]:
df = pd.read_csv("IMDB Dataset.csv")

## 3. Data Checks and Cleaning

Before modelling, we look at the size of the data, a few sample rows, missing values and duplicates.

**Dataset size** – the shape shows (number of rows, number of columns).

In [34]:
df.shape

(50000, 2)

**Preview** – the first five rows of the dataset.

In [35]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


**Missing values** – count of empty cells in each column. Zero means the data is complete.

In [36]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

**Remove duplicate reviews** – duplicates can bias the model, so we drop them and check the new size.

In [37]:
df.drop_duplicates(inplace=True)

In [38]:
df.shape

(49582, 2)

**Class balance** – a quick chart showing how many positive and negative reviews we have. A balanced dataset means the model is not biased towards one class.

In [ ]:
# Visual: number of reviews per sentiment class
df["sentiment"].value_counts().plot(kind="bar", color=["#4C78A8", "#E45756"])
plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

## 4. Text Pre-processing

Raw reviews contain noise (capital letters, links, HTML tags, symbols, very common words). Cleaning the text helps the model focus on the words that carry meaning.

The steps below are applied in this order:

`Lowercase → Remove URLs → Remove punctuation → Remove HTML → Remove stopwords → Stemming → Encoding → Vectorization`

### 4.1 Convert Text to Lowercase

Makes `Movie`, `MOVIE` and `movie` the same word.

In [39]:
df["review"] = df["review"].str.lower()

### 4.2 Remove URLs

Web links do not help in finding the sentiment, so we remove anything that starts with `http`.
The function uses a regular expression with `re.sub(pattern, replacement, text)`.

In [41]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 4.3 Remove Punctuation

Keeps only letters, numbers and spaces, and removes every other symbol (`! ? , . " '` etc.).

In [42]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [43]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4.4 Remove HTML Tags

Reviews scraped from the web contain tags like `<br />`. This step removes everything written between `<` and `>`.

In [44]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 4.5 Remove Stopwords

Stopwords are very common words such as *the, is, and, of* that add little meaning.
We first download the NLTK resources, then tokenize each review (split it into words) and remove the stopwords.

> **Note:** The function removes stopwords by replacing them directly inside the text, so short stopwords can also be removed from inside longer words (for example `reviewers` becomes `revewers` in the output below). The model still trains on this text; this is a good point to improve in a future version.

In [45]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/apnacollege/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/apnacollege/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/apnacollege/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [48]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

Preview of the reviews after stopword removal.

In [49]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 4.6 Stemming

Stemming reduces words to their root form so that similar words are treated as one, for example `running → run` and `played → play`.
We use NLTK's **Porter Stemmer**.

In [51]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [52]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 4.7 Label Encoding

Machine learning models need numbers, not text. The target column is converted as:

- `negative → 0`
- `positive → 1`

The encoded column is stored in `y` (our target variable).

In [53]:
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [54]:
y = df["sentiment"]

In [55]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 4.8 Vectorization (TF-IDF)

TF-IDF turns each review into a numeric vector. A word gets a higher score when it is frequent in a review but rare across all reviews.

We keep the **top 5,000 words** (`max_features=5000`), so each review becomes a vector of 5,000 numbers stored in `X` (our input features).

In [56]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [57]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## 5. Train/Test Split and Data Loaders

### 5.1 Split the Data

- **80%** of the data is used for training
- **20%** is kept aside for testing the model on unseen reviews
- `random_state=42` makes the split reproducible

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Shape of the training data (rows, features).

In [72]:
X_train.shape

(39665, 5000)

Shape of the testing data (rows, features).

In [73]:
X_test.shape

(9917, 5000)

### 5.2 Create PyTorch Datasets and Data Loaders

- The sparse TF-IDF matrices are converted to regular arrays
- Arrays are converted to PyTorch tensors and wrapped in a `TensorDataset`
- A `DataLoader` feeds the data to the model in **mini-batches of 64** reviews (shuffled)

In [75]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [76]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [77]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## 6. Build the RNN Model

### What is an RNN?
A Recurrent Neural Network (RNN) is a neural network designed for sequential data such as text. It keeps a *hidden state* (a memory) that is passed from one step to the next.

### Our model
| Part | Description |
|---|---|
| `nn.RNN` | RNN layer with **128 hidden units** and 1 layer |
| `nn.Linear` | Fully connected layer that converts the final hidden state into **1 output** (the sentiment score) |

Each review enters the model as a TF-IDF vector of 5,000 values, so `input_size = 5000`.

In [81]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

### Loss Function and Optimizer

- **Loss:** `BCELoss` (Binary Cross-Entropy), the standard loss for two-class problems
- **Optimizer:** `Adam`, which updates the weights during training

In [82]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## 7. Train the Model

The model is trained for **10 epochs** (10 full passes over the training data). In each step:

1. Clear old gradients (`zero_grad`)
2. Run a forward pass to get predictions
3. Apply `sigmoid` to turn the output into a probability between 0 and 1
4. Compute the loss
5. Backpropagate (`loss.backward`) and update the weights (`optimizer.step`)

> The loss printed after each epoch is the loss of the **last mini-batch** of that epoch, so it can go up and down between epochs.

In [88]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.14945334196090698
epoch = 2/10 and loss = 0.15595930814743042
epoch = 3/10 and loss = 0.264388769865036
epoch = 4/10 and loss = 0.26949164271354675
epoch = 5/10 and loss = 0.3557273745536804
epoch = 6/10 and loss = 0.13662415742874146
epoch = 7/10 and loss = 0.1333741396665573
epoch = 8/10 and loss = 0.13024254143238068
epoch = 9/10 and loss = 0.33016765117645264
epoch = 10/10 and loss = 0.3183642327785492


## 8. Evaluate the Model

The model is switched to evaluation mode and tested on the unseen test set. A review is predicted as **positive** when its probability is greater than 0.5.

**Accuracy** = correct predictions ÷ total predictions × 100

In [89]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.15680145205204


## 9. Conclusion

- An RNN classifier was built in PyTorch to predict the sentiment of IMDB movie reviews.
- Reviews were cleaned and converted to numbers with TF-IDF (5,000 features).
- The model reached about **85.16% accuracy** on the unseen test data.

### Ideas for improvement
- Compare with **LSTM** or **GRU**, which handle long-term context better than a basic RNN
- Use word embeddings and feed reviews as real word sequences instead of one TF-IDF vector
- Use a proper stopword removal method that works on whole words
- Track the loss and accuracy for every epoch and plot the learning curves
- Add a validation set and try hyperparameter tuning